# PROYECTO SPRINT 15 - RUSTY BARGAIN
El servicio de venta de autos usados Rusty Bargain está desarrollando una aplicación para atraer nuevos clientes. Gracias a esa app, puedes averiguar rápidamente el valor de mercado de tu coche. Tienes acceso al historial: especificaciones técnicas, versiones de equipamiento y precios. Tienes que crear un modelo que determine el valor de mercado.
A Rusty Bargain le interesa:
- la calidad de la predicción;
- la velocidad de la predicción;
- el tiempo requerido para el entrenamiento

## 1. Preparación de datos

### 1.1 Importar librerias y cargar datos

In [1]:
# Librerias principales
import pandas as pd
import numpy as np
import time

# Librerias para los modelos
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Librerias de potenciación de descenso de gradiente
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

# Definición del random state
rnd_stt = 792604

# Opciones de visualización
pd.set_option('display.max_columns', None)

# Carga de datos
df = pd.read_csv('/datasets/car_data.csv')

print(df.shape)
df.head()

(354369, 16)


,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,Brand,NotRepaired,DateCreated,NumberOfPictures,PostalCode,LastSeen
0,24/03/2016 11:52,480,NaN,1993,manual,0,golf,150000,0,petrol,volkswagen,NaN,24/03/2016 00:00,0,70435,07/04/2016 03:16
1,24/03/2016 10:58,18300,coupe,2011,manual,190,NaN,125000,5,gasoline,audi,yes,24/03/2016 00:00,0,66954,07/04/2016 01:46
2,14/03/2016 12:52,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,NaN,14/03/2016 00:00,0,90480,05/04/2016 12:47
3,17/03/2016 16:54,1500,small,2001,manual,75,golf,150000,6,petrol,volkswagen,no,17/03/2016 00:00,0,91074,17/03/2016 17:40
4,31/03/2016 17:25,3600,small,2008,manual,69,fabia,90000,7,gasoline,skoda,no,31/03/2016 00:00,0,60437,06/04/2016 10:17


#### Observaciones
El juego de datos contiene 354369 entradas u observaciones y cuenta con 16 características de acuerdo con el siguiente listado:
- DateCrawled: Fecha en la que se descargó el perfil de la base de datos.
- Price: Precio del vehiculo, (Euros).
- VehicleType: Tipo de carrocería del vehículo.
- RegistrationYear: Año de matriculación del vehículo.
- Gearbox: Tipo de caja de cambios.
- Power: Potencia (CV).
- Model: Modelo del vehículo.
- Mileage: Kilometraje (medido en km de acuerdo con las especificidades regionales del conjunto de datos).
- RegistrationMonth: Mes de matriculación del vehículo.
- FuelType: Tipo de combustible.
- Brand: Marca del vehículo.
- NotRepaired: Vehículo con o sin reparación.
- DateCreated: Fecha de creación del perfil.
- NumberOfPictures: Número de fotos del vehículo.
- PostalCode: Código postal del propietario del perfil (usuario).
- LastSeen: Fecha de la última vez que el usuario estuvo activo.

### Exploración inicial de los datos

In [2]:
# Información general datos
print(df.info())
print()

# Datos nulos
print(df.isna().sum().sort_values(ascending=False))

# Datos estadísticos básicos
df.describe().T



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354369 entries, 0 to 354368
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   DateCrawled        354369 non-null  object
 1   Price              354369 non-null  int64 
 2   VehicleType        316879 non-null  object
 3   RegistrationYear   354369 non-null  int64 
 4   Gearbox            334536 non-null  object
 5   Power              354369 non-null  int64 
 6   Model              334664 non-null  object
 7   Mileage            354369 non-null  int64 
 8   RegistrationMonth  354369 non-null  int64 
 9   FuelType           321474 non-null  object
 10  Brand              354369 non-null  object
 11  NotRepaired        283215 non-null  object
 12  DateCreated        354369 non-null  object
 13  NumberOfPictures   354369 non-null  int64 
 14  PostalCode         354369 non-null  int64 
 15  LastSeen           354369 non-null  object
dtypes: int64(7), object(

,count,mean,std,min,25%,50%,75%,max
Price,354369.0,4416.656776,4514.158514,0.0,1050.0,2700.0,6400.0,20000.0
RegistrationYear,354369.0,2004.234448,90.227958,1000.0,1999.0,2003.0,2008.0,9999.0
Power,354369.0,110.094337,189.850405,0.0,69.0,105.0,143.0,20000.0
Mileage,354369.0,128211.172535,37905.341530,5000.0,125000.0,150000.0,150000.0,150000.0
RegistrationMonth,354369.0,5.714645,3.726421,0.0,3.0,6.0,9.0,12.0
NumberOfPictures,354369.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
PostalCode,354369.0,50508.689087,25783.096248,1067.0,30165.0,49413.0,71083.0,99998.0


#### Observaciones:
En general se observa un juego de datos incompleto en algunas características que requieren un procedimiento de limpieza antes de ser procesados. Esto se nota en las características 'NotRepaired' con 71154 valores faltantes,'VehicleType' con 37490 valores ausentes, 'FuelType' cpm 32895 valores faltantes, 'GearBox' con 19833 datos y finalmente con 'Model' que le faltan 19705 datos.
Todas las anteriores características son requeridas para el entrenamiento de los modelos, ya que sus valores van a afectar las predicciones del modelo.

En los datos estadísticos básicos también se encuentran situaciones que requieren atención:
- La característica 'RegistrationYear' tiene un valor minimo de 1000 y un valor máximo de 9999, valor totalmente atípico para el manejo de los datos.
- en 'Power' tambien se observa que el valor promedio es de 100 CV pero hay un valor máximo de 20000 y un valor mínimo de 0, datos que también son atípicos.
- En 'RegistrationMonth' se tiene un valor mínimo de 0, lo cual sería un valor atípico teniendo en cuenta que son 12 meses del año.
- Las columnas de 'NumbreOfPictures' y 'PostalCode' son datos que no aportan información valiosa al modelo, también el número de fotografías se mantiene en 0 para todo el juego de datos, posiblemente al formar la base de datos se montó esta casilla, pero nunca se diligenció adecuadamente.

Por último, las caracetrísticas asociadas a fechas son datos valiosos para trazabilidad web, pero no aportan información sobre el vehículo y no deben usarse en los entrenamientos de los modelos.

### Limpieza de datos

In [3]:
# Remoción de columnas irrelevantes para el modelo de predicción de precio del vehículo
columns_to_drop = ['NumberOfPictures', 'PostalCode', 'DateCrawled', 'DateCreated', 'LastSeen']
df = df.drop(columns_to_drop, axis=1)

# Corroboración
df.shape

(354369, 11)

In [4]:
# Remoción datos atípicos
# 'Price' debe ser mayor a 0
# 'Power' debe tener datos mayores a 0 y menores a 1000
# 'RegistrationYear' debe tener datos mayores a 1900 y menores a 2016 (Fecha del dataset)
# 'RegistrationMonth' debe tener datos mayores a 0

# Datos antes de la limpieza
rows_before = len(df)

# Criterios de remoción
mask = ((df['Price'] > 0) & (df['Power'] >= 1) & (df['Power'] <= 1000) & (df['RegistrationYear'] >= 1900) & (df['RegistrationYear'] <= 2016) & (df['RegistrationMonth'] > 0))

# Remoción
df = df[mask].reset_index(drop=True)

# Datos posteriores a la limpieza
rows_after = len(df)

print(f'Filas antes: {rows_before}')
print(f'Filas después: {rows_after}')
print(f'Eliminadas: {rows_before - rows_after} ({(rows_before - rows_after) / rows_before:.1%})')

Filas antes: 354369
Filas después: 278636
Eliminadas: 75733 (21.4%)


#### Observaciones
Se han retirado 75733 datos con valores atipicos para precio, año y mes de registro y potencia. Si bien esto representa el 21% de los datos iniciales del dataset, si se conservan estos datos el modelo podría sesgarse, además para el entrenamiento podrán quedar un poco más de 278000 observaciones, número suficiente para las etapas de entrenamiento.

In [5]:
# Tratamiento características categóricas

categorical_columns = ['VehicleType', 'Gearbox', 'Model', 'FuelType', 'Brand', 'NotRepaired']

for col in categorical_columns:
    df[col] = df[col].fillna('unknown')

#Comprobación del tratamiento
df.isna().sum()

Price                0
VehicleType          0
RegistrationYear     0
Gearbox              0
Power                0
Model                0
Mileage              0
RegistrationMonth    0
FuelType             0
Brand                0
NotRepaired          0
dtype: int64

#### Observaciones

Aprovechando que los métodos de potenciación de descenso de gradiente pueden manejar datos categóricos, y que los valores ausentes en estas características pueden llegar a ser numerosos, se hizo el cambio a 'Unknown'.

Se realiza la corroboración de estos datos y se encuentra que ya no hay valores ausentes como tal.

In [6]:
# Revisión filas duplicadas
print(f'Filas duplicadas: {df.duplicated().sum()}')

# Remoción datos duplicados
df = df.drop_duplicates().reset_index(drop=True)

#Forma final dataset
print(f'Forma del Dataset limpio: {df.shape}')

rows_final = len(df)

print(f'Filas antes: {rows_before}')
print(f'Filas finales: {rows_final}')
print(f'Eliminadas: {rows_before - rows_final} ({(rows_before - rows_final) / rows_before:.1%})')

df.describe().T

Filas duplicadas: 22434
Forma del Dataset limpio: (256202, 11)
Filas antes: 354369
Filas finales: 256202
Eliminadas: 98167 (27.7%)


,count,mean,std,min,25%,50%,75%,max
Price,256202.0,4999.311680,4658.414704,1.0,1400.0,3350.0,7200.0,20000.0
RegistrationYear,256202.0,2002.942850,6.505678,1910.0,1999.0,2003.0,2007.0,2016.0
Power,256202.0,121.289744,54.908323,1.0,79.0,113.0,150.0,1000.0
Mileage,256202.0,127767.445219,36937.734956,5000.0,125000.0,150000.0,150000.0,150000.0
RegistrationMonth,256202.0,6.372593,3.345036,1.0,4.0,6.0,9.0,12.0


#### Observaciones
Se encontraron 22434 datos duplicados. Esto puede deberse a problemas durante la inscripción de los datos. Como son datos de autos usados, es poco probable que todos los datos que se encuentran duplicados correspondan a diferentes entradas, por otro lado, incluir estos datos duplicados en el entrenamiento puede llevar a sesgos en los modelos obtenidos.

Finalmente se eliminan en total 98167 datos incluyendo valores duplicados y valores atípicos. Esto representa casi el 28% de los datos originales, sin embargo, quedan disponibles 256202 observaciones, lo cual es un número apropiado para obtener un modelo confiable.

### 1.4 Formación conjuntos de entrenamiento y prueba, con características y objetivo

In [7]:
# Objetivo
target = df['Price']
# Características
features = df.drop(['Price'], axis=1)

features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.25, random_state=rnd_stt)

print('Entrenamiento:', features_train.shape)
print('Prueba:', features_test.shape)

Entrenamiento: (192151, 10)
Prueba: (64051, 10)


#### Observaciones
Se realiza la partición de datos para entrenamiento y prueba.
- Se forma un conjunto de entrenamiento con 192151 observaciones, lo cual es un valor suficiente para entrenar diferentes algoritmos.
- Para la prueba se contará con un juego de datos de 64051 observaciones, lo que nos permitirá corroborar la validez de los modelos obtenidos.
- En mabos casos, se separa la característica 'Price' como objetivo de las predicciones.

## 2. Entrenamiento del modelo 

### 2.1 Codificación de características categóricas

Dado que se deben entrenar varios algoritmos, se requiere el tratamiento adecuado para las características aleatorias que se van a manejar.

- **Regresión lineal, árbolde deceisión, bosque aleatorio y XGBoost:** como estos algoritmos no manejan texto de forma nativa, se realizará *codificación OHE*.
- **LightGBM:** Este algoritmo soporta características categóricas de forma nativa. *No requiere OHE.*
- **CatBoost:** Soporta características categóricas como *strings* al indicársele que lo son mediante 'cat_features'. *No requiere OHE*. 

In [8]:
# Codificación OHE
features_train_ohe = pd.get_dummies(features_train, columns=categorical_columns, drop_first=True)
features_test_ohe = pd.get_dummies(features_test, columns=categorical_columns, drop_first=True)

# asegurar que ambos juegos de datos tengan el mismo número de columnas 
features_train_ohe, features_test_ohe = features_train_ohe.align(features_test_ohe, join='left', axis=1, fill_value=0)
# Revisión conformación juegos de datos luego de aplicar OHE
print('Forma OHE (train):', features_train_ohe.shape)
print()
print('Forma OHE (test):', features_test_ohe.shape)

Forma OHE (train): (192151, 311)

Forma OHE (test): (64051, 311)


#### Observaciones:
Se usa el método OHE para el tratamiento de características categóricas.
Adicional a ello, se realiza un paso para asegurar que todas las columnas del juego de entrenamiento se encuentren en el juego de prueba, dado que inicialmente no eran las mismas. Como el juego de entrenamiento tiene 3 veces mas datos que el de prueba, estamos garantizando que todas tengan el mismo número de columnas.

In [9]:
# Codificación 'category' para el algoritmo LightGBM
features_train_cat = features_train.copy()
features_test_cat = features_test.copy()

for col in categorical_columns:
    features_train_cat[col] = features_train_cat[col].astype('category')
    features_test_cat[col] = features_test_cat[col].astype('category')
    # alineamos las categorías entre train y test para evitar categorías desconocidas
    features_test_cat[col] = features_test_cat[col].cat.set_categories(features_train_cat[col].cat.categories)

features_train_cat.dtypes

VehicleType          category
RegistrationYear        int64
Gearbox              category
Power                   int64
Model                category
Mileage                 int64
RegistrationMonth       int64
FuelType             category
Brand                category
NotRepaired          category
dtype: object

#### Observaciones
En este caso, los juegos de datos se llevan a forma 'category' para poder ser utilizados en el algoritmo LightBCM. Se realiza la corroboración y se encuentra que las características categóricas se encuentran en la forma deseada.

In [10]:
# Codificación para algoritmo CatBoost
features_train_pool = features_train.copy()
features_test_pool = features_test.copy()

cat_features_idx = [features_train_pool.columns.get_loc(col) for col in categorical_columns]
print('Índices de columnas categóricas:', cat_features_idx)

Índices de columnas categóricas: [0, 2, 4, 7, 8, 9]


#### Observaciones
En este último caso, la codificación es especificarle al algoritmo que características serán categóricas para que pueda aplicar el modelo adecuadamente. 

Se revisa y las columnas categóricas coinciden con los índices establecidos.

### 2.2 Función reutilizable de evaluación de modelos

Se monta una función de evaluación de modelos con las métricas solicitadas de RECM y los tiempos de computo.

In [11]:
# Tabla de resultados
results = []

# Función de evaluación
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name, params=''):
    
# Ajuste del entrenamiento
    start_fit = time.time()
    model.fit(X_train, y_train)
    # Tiempo de entrenamiento
    train_time = time.time() - start_fit

    # Predicciones
    start_pred = time.time()
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    predict_time = time.time() - start_pred


    # Evaluación RMSE
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    rmse_test = mean_squared_error(y_test, pred_test) ** 0.5

    # Impresión resultados
    print(f'{model_name} - {params}')
    print(f'RECM del entrenamiento: {rmse_train:.1f}')
    print(f'RECM de la prueba: {rmse_test:.1f}')
    print(f'Tiempo de entrenamiento: {train_time:.2f}')
    print(f'Tiempo de prueba: {predict_time:.2f}')

    # Generación resultados en tabla
    results.append({'Modelo': model_name, 'Hiperparámetros': params, 'RECM_train': round(rmse_train, 1), 'RECM_test': round(rmse_test, 1), 'Tiempo_entrenamiento': round(train_time, 2), 'Tiempo_predicción': round(predict_time, 2)})

    return model


#### Observaciones
Se genera una función para la evaluación de los modelos a entrenar, con el objetivo de reducir codigo redundante ya que los entrenamientos y las predicciones manejan basicamente el mismo conjunto de datos, solo que se cambiaría el nombre de acuerdo al algoritmo utilizado.
Aunque el proyecto recomienda usar un comando especial de jupyter notebooks para la evaluación del tiempo, si quiero almacenar el tiempo en una tabla comparativa quedaría información irrelevante para el análisis.

### 2.3 Regresión lineal - Prueba de cordura
Se toma la regresión lineal como prueba de cordura, es decir, sus resultados servirán como línea base de comparación con los demás modelos a entrenar.

In [12]:
# Regresión lineal
_ = evaluate_model(LinearRegression(), features_train_ohe, target_train, features_test_ohe, target_test, 'Regresión Lineal')

Regresión Lineal - 
RECM del entrenamiento: 2614.8
RECM de la prueba: 2607.5
Tiempo de entrenamiento: 6.78
Tiempo de prueba: 0.58


#### Observaciones
- Los valores de RECM entre el entrenamiento y la validación es muy pequeña, esto indica que el modelo tiene una precisión similar con datos conocidos (entrenamiento) y datos nuevos (prueba).
- Sin embargo, teniendo en cuenta que el precio promedio de venta de coches es de 5000, un RECM de 2600 nos presenta un error relativo superior al 50%, lo que representa una alta incertidumbre en la predicción.
- Finalmente, como la regresión representará el modelo de cordura, se espera que los siguientes modelos a desarrollar presenten valores de RECM menores a este, y que tenga similitud entre los datos de entrenamiento y de prueba.
- Sobre la velocidad o tiempo que toma el modelo en entrenarse y predecir, se puede concluir que el modelo de regresión lineal es un modelo rápido en la predicción, tomando menos de 1 segundo en realizar predicciones. Esto es un valor importante para el cliente.
- Se puede concluir entonces que **El modelo de Regresión lineal es un método rápido pero poco preciso.**

### 2.4 Árbol de decisición
Para este algoritmo se tomarán profundidades de árbol de 8 y 12, evaluando como se comporta el RECM y los tiempos de computo.

In [13]:
%%time
# Árbol de decisión

for depth in [8,12]:
    _ = evaluate_model(DecisionTreeRegressor(max_depth=depth, random_state=rnd_stt), features_train_ohe, target_train, features_test_ohe, target_test, 'Árbol de decisión', params=f'(max_depth={depth})')

Árbol de decisión - (max_depth=8)
RECM del entrenamiento: 2081.4
RECM de la prueba: 2100.0
Tiempo de entrenamiento: 2.10
Tiempo de prueba: 0.29
Árbol de decisión - (max_depth=12)
RECM del entrenamiento: 1731.9
RECM de la prueba: 1918.3
Tiempo de entrenamiento: 2.77
Tiempo de prueba: 0.31
CPU times: user 4.91 s, sys: 667 ms, total: 5.58 s
Wall time: 5.48 s


#### Observaciones

Se entrenan modelos de árbol de decisión a partir de 2 profundidades de árbol, 8 y 12, con el fin de evaluar las métricas de RECM, los tiempos de computo bajo distintas complejidades basadas en la profundidad.

*Árbol de profundidad 8:*
- Comparado con la prueba de cordura, encontramos que el RECM del entrenamiento y la prueba se reduce sin perder la precisión entre uno y otro. Esto nos indica que el modelo funciona adecuadamente en la predicción con un mejor comportamiento que la prueba de cordura.
- En cuanto a tiempos también se advierte una mejora respecto a entrenamiento y predicción comparado con la regresión lineal, es un **método más rápido y más preciso que la prueba de cordura.**

*Árbol de profundidad 12:*
- Los RECM tanto de entrenamiento y prueba bajan de la barrera de los 2000, lo cual representa un 40% de error del precio medio de los vehiculos. Si bien aún se puede mejorar mucho en la predicción, el modelo empieza a mejorar la precisión.
- Se debe tener en cuenta que la brecha entre el RECM del entrenamiento y prueba empieza a ampliarse en casi 186 unidades, esto significa que el modelo empieza a presentar algo de *Overfitting* probablemente por la complejidad del algoritmo. Sin embargo, esta brecha todavía puede aceptarse teniendo en cuenta la precisión.
- Nuevamente, los tiempos de entrenamiento y prueba son pequeños comparados con el modelo de cordura, mostrando que una mayor profundidad del árbol no afecta de manera drástica el tiempo de computo.
- Este algoritmo de árbol de decisión de profundidad 12 es **preciso y rápido, con un ligero sobreajuste de los datos.**

### 2.5 Bosque aleatorio
Se realizará la evaluación de n_estimators entre 50 y 100, y se evaluará la profundidad entre 10 y 12 para evaluar el comportamiento de las métricas establecidas.

In [ ]:
# Bosque aleatorio
for n_estimators, depth in [(50, 10), (100, 12)]:
    _ = evaluate_model(RandomForestRegressor(n_estimators=n_estimators, max_depth=depth, random_state=rnd_stt, n_jobs=-1), features_train_ohe, target_train, features_test_ohe, target_test, 'Bosque aleatorio', params=f'(n_estimators={n_estimators}, max_depth={depth})')



#### Observaciones

Se entrenan modelos de bosque aleatorio con los siguientes hiperparámetros:

*Bosque de 50 árboles y profundidad 10:*
- El valor de RECM tanto de entrenamiento y prueba es menor a 1900, valores menores que el modelo de árbol de decisión de profundidad 12. Seguimos mejorando la precisión a medida que aumentamos la complejidad.
- La brecha de RECM entre el entrenamiento y la prueba es de 75 unidades, lo que muestra un *Overfitting* bajo.
- El tiempo de respuesta en el entrenamiento aumenta por la complejidad del algoritmo a 35 segundos, pero es un tiempo moderado.
- Se puede concluir que este algoritmo es **preciso y rápido.**

*Bosque de 100 árboles y profundidad 12:*
- La precisión de este algoritmo mucho mas complejo sigue aumentando, ya se reduce el valor del RECM a 1633 en entrenamiento y 1785 en prueba, valores entre el 33 y35% del precio medio de los vehículos. Un resultado notable hasta el momento.
- El *Overfitting* sigue siendo mesurado, incluso menor que el obtenido con un solo árbol. La presencia de muchos árboles ayuda a evitar esta situación.
- El tiempo de computo en entrenamiento y prueba es el coste actual del modelo, aumenta a 78 segundos en el entrenamiento y casi 1.5 segundos en la prueba. Esto se debe al alto número de árboles sobre los que se tiene que trabajar.
- Este algoritmo es entonces **preciso, con bajo sobreajuste, pero algo lento.**


### 2.6 LightGBM
De acuerdo a la teoria y documentación de este método, ya no se requiere la codificación OHE desarrollada en los algoritmos anteriores.

Se probarán 2 configuraciones diferentes de hiperparámetros para la evaluación.

In [ ]:
# Algoritmo LightGBM

#Configuraciones a evaluar
lgb_cfg = [{'num_leaves': 31, 'learning_rate': 0.1, 'n_estimators': 200}, {'num_leaves': 63, 'learning_rate': 0.05, 'n_estimators': 400}]

for cfg in lgb_cfg:
    model = lgb.LGBMRegressor(random_state=rnd_stt, verbose=-1, **cfg)

    _ = evaluate_model(model, features_train_cat, target_train, features_test_cat, target_test, 'LightGBM', params=str(cfg))

#### Observaciones

Se realiza el entrenamiento con el algoritmo Light_GBM, hubo que apoyarse un poco en la documentación y en IA para entenderlo con algo de profundidad. Se evaluan 2 configuraciones:

*Primera configuración: leaves = 31, learning rate = 0.1, n_estimators = 200*
- Esta es una configuración *conservadora*, donde se toman 'número de hojas por árbol' de 31 hojas, con una tasa de aprendizaje 'rápida' de 0.1, y con 200 árboles.
- La precisión sigue aumentando con una baja brecha entre entrenamiento y prueba, es decir se tiene un algoritmo preciso y con bajo *Overfitting*. la precisión del entrenamiento ya es cercana al 31%.
- Los tiempos de entrenamiento y prueba son bajos a nivel de segundos, demostrando la potencia de estos algoritmos de descenso de gradiente en el tiempo de computo.
- Este es un **algoritmo preciso y rápido, con bajo overfitting.**

*segunda configuración: leaves = 63, learning rate = 0.05, n_estimators = 400*
- Esta es una configuración mas *compleja*, donde se toman 'número de hojas por árbol' de 63 hojas, con una tasa de aprendizaje 'lenta' de 0.05, y con 400 árboles. 
- La métrica RECM en entrenamiento ya es menor al 30% relativo al precio medio de los vehículos con un *Overfitting* mesurado entre el entrenamiento y la prueba.
- Los tiempos de computo aumentan en comparación con la primera configuración, pero siguen siendo bajos.
- Este algoritmo viene siendo **preciso, ligeramente lento, con Overfitting moderado.**

### 2.7 Catboost
Nuevamente, se definen 2 configuraciones de evaluación

In [ ]:
# Algoritmo CatBoost

#Configuraciones a evaluar
catboost_cfg = [{'iterations': 300, 'depth': 6, 'learning_rate': 0.1}, {'iterations': 500, 'depth': 8, 'learning_rate': 0.05}]

for cfg in catboost_cfg:
    model = CatBoostRegressor(random_state=rnd_stt, cat_features=cat_features_idx, verbose=False, **cfg)

    _ = evaluate_model(model, features_train_pool, target_train, features_test_pool, target_test, 'CatBoost', params=str(cfg))


#### Observaciones

Se realiza el entrenamiento con el algoritmo CatBoost, nuevamente realizando 2 configuraciones que buscan abarcar un espectro amplio del algoritmo:
- La primera realiza 300 iteraciones con árboles de profundidad 6 y con un tiempo de aprendizaje moderado.
- La segunda lleva el número de iteraciones a 500, aumenta ligeramente la profundidad a 8, y reduce la tasa de aprendizaje para compensar. Esta configuración es más compleja por el número de iteraciones que realiza.

*Primera configuración: iterations = 300, learning rate = 0.1, depth = 6*
- Bajo esta configuración el RECM del algoritmo no mejora los resultados obtenidos con el algoritmo *LightGBM*, el error relativo es superior al 30%, aunque la brecha entre las predicciones del entrenamiento y la prueba se reducen considerablemente.
- El tiempo de entrenamiento es mayor que el algoritmo anterior, debido principalmente al número de iteraciones, pero el tiempo de prueba es realmente pequeño.
- Este es un **algoritmo preciso, moderadamente rápido, sin overfitting.**

*Segunda configuración: iterations = 500, learning rate = 0.05, depth = 8*
- Al ser una configuración con mayor complejidad, el RECM se reduce pero no alcanza los valores obtenidos con el algoritmo *LightGBM*.
- El *Overfitting* sigue siendo muy bajo entre el entrenamiento y la prueba, mostrando una solidez del modelo para realizar predicciones con datos nuevos.
- El tiempo de computo aumenta considerablemente debido al número de iteraciones que realiza, pero el tiempo de predicción es muy bajo.
- Este algoritmo viene siendo **preciso, lento, con Overfitting moderado.**

Si bien este algoritmo de potenciación del descenso de gradiente no obtiene mejoras considerables que el algoritmo *LightGBM*, tiene mejores métricas que los modelos convencionales vistos anteriormente. 


### 2.8 XGBoost
Para utilizar este algoritmo, se debe utilizar nuevamente la codificación OHE, formulando 2 configuraciones diferentes.

In [ ]:
# Algoritmo XGBoost

# Configuraciones a evaluar
xgb_cfg = [{'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1}, {'n_estimators': 400, 'max_depth': 8, 'learning_rate': 0.05}]

for cfg in xgb_cfg:
    model = xgb.XGBRegressor(random_state=rnd_stt, n_jobs=-1, **cfg)
    evaluate_model(model, features_train_ohe, target_train, features_test_ohe, target_test, 'XGBoost', params=str(cfg))

#### Observaciones

Se monta el algoritmo XGBoost, bajo 2 configuraciones que siguen el mismo patrón que se ha desarrollado con algoritmos previos, cubrir un espectro de opciones que permitan comparar métricas.
- La primer configuración de 200 estimadores, de profundidad 6 y con tasa de aprendizaje de 0.1, como modelo conservador.
- La segunda lleva los estimadores a 400, con profundidad de árbol de 8 y tasa de aprendizaje de 0.05, que compense el efecto de los otros parámetros.

*Primera configuración:*
- La métrica de RECM para este algoritmo sigue siendo buena comparada con la prueba de cordura, pero tiene un valor menor que el obtenido con LightGBM y es ligeramente superior que el obtenido con CatBoost. La brecha que tienen las RECM de entrenamiento y prueba es pequeña, indicando un buen desarrollo como se ha visto con todos los algoritmos asociados a la potenciación del descenso de gradiente.
- El tiempo de computo por otro lado es alto, llegando a casi los 5 minutos de computo en el entrenamiento y casi 2 segundos en la prueba. Este alto tiempo es un valor desfavorable en comparación con los otros algoritmos estudiados.
- En resumen, este es un **algoritmo preciso, con bajo Overfitting, pero de baja velocidad de computo.**

*Segunda configuración*: 
- La configuración de alta complejidad muestra un RECM cercano al obtenido con LightGBM. Con un error relativo inferior al 30%.
- La brecha de las predicciones de entrenamiento y prueba son moderadas, pero dentro de valores normales a los obtenidos en todos los algoritmos estudiados con una complejidad media-alta.
- Finalmente, este algoritmo ha presentado los mayores tiempos de computo de todos los algoritmos analizados. Con un tiempo de entrenamiento cercano a los 13 minutos, valor muy alto para los requerimientos del cliente. Para el grupo de prueba el tiempo se reduce a poco mas de 3 segundos.
- Este algoritmo viene siendo **preciso, muy lento, con Overfitting moderado.**

## 3. Análisis del modelo

### 3.1 Consolidación de la información

In [ ]:
results_df = pd.DataFrame(results).sort_values('RECM_test').reset_index(drop=True)
results_df

### 3.2 Comparación visual

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(15,15))

# Grafico de barras para calidad del modelo
axes[0].barh(results_df['Modelo']+ ' ' + results_df['Hiperparámetros'], results_df['RECM_test'])
axes[0].set_xlabel('RECM del modelo (menor es mejor)')
axes[0].set_title('Calidad del modelo en las predicciones')
axes[0].invert_yaxis()

# Gráfico de barras para el tiempo de computo
axes[1].barh(results_df['Modelo']+ ' ' + results_df['Hiperparámetros'], results_df['Tiempo_entrenamiento'])
axes[1].set_xlabel('Tiempo de entrenamiento (s)')
axes[1].set_title('Velocidad de entrenamiento')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

#### Observaciones
**Observaciones de calidad del algoritmo**
En general se puede determinar que:
- **Prueba de cordura:** Sirve como línea base para analizar los resultados de los algoritmos establecidos en el proyecto. Presenta una precisión con un error relativo superior al 50%, con un bajo *Overfitting*.
- **Árbol de decisión vs Bosque aleatorio:** Ambos algoritmos se probaron con 2 diferentes juegos de hiperparámetros. En términos generales se concluye que una mayor complejidad en el algoritmo reduce el RECM, pero aumenta de alguna forma la brecha entre los juegos de entrenamiento y de prueba, aumentando ligeramente el *Overfitting*. Como ya se ha observado en otro momento, un bosque aleatorio presentará un mejor comportamiento al considerar un gran número de árboles.
- **Modelos de potenciación de gradiente:** En los 3 métodos analizados se obtuvieron mejores resultados de RECM en los modelos con mayor magnitud de sus hiperparámetros. Los métodos de potenciación incluso presentan mejores calidades que el método convencional de *Bosque aleatorio*, esto debido a la naturaleza de los métodos de incluir un residuo en su iteración anterior para ir minimizando el error. 

**Observaciones de tiempo de entrenamiento**
- **Regresión lineal:** Representa el tiempo base sobre el cual se realizanrán las comparaciones de los demás modelos obtenidos.
- **Árbol de decisión vs Bosque aleatorio:** Ambos algoritmos se probaron con 2 configuraciones diferentes para analizar tanto su calidad como tiempo de procesamiento. Como era de esperarse, el bosque aleatorio al incluir más 'árboles' en su implementación, toma un tiempo mayor en su entrenamiento. No obstante, son tiempos de entrenamiento moderados de unos cuantos segundos y no deberían representar un problema serio al momento de implementarse en una aplicación.
- **Modelos de potenciación de gradiente:** Acá es probablemente donde mayores diferencias se observan. Los algoritmos CatBoost y XGBoost tienen altos tiempos de entrenamiento, especialmente si los hiperparámetros propios del algoritmo le dan una mayor complejidad. El método LightGBM por su parte tiene tiempos muy favorables de entrenamiento, incluso con una alta complejidad.
- En todos los casos, los tiempos de predicción con datos de prueba mostraron tiempos de procesamiento cortos y no deben generar problemas en su implementación final. 

## Conclusiones:
- Se desarrollaron diferentes modelos de Machine Learning para la predicción de precio de vehículos usados, a partir de un gran dataset disponible con varias características del vehículo, tanto numéricas como categóricas.
- Los datos se sometieron a un tratamiento especial para eliminar datos duplicados, información irrelevante, y datos atípicos que pudieran afectar la veracidad de los resultados de predicción obtenidos.
- Las características categóricas se sometieron a un tratamiento de codificación para poder realizar las regresiones requeridas.
- Teniendo en cuenta que se contaba con cerca de 10 características, muchas de ellas categóricas, la Regresión lineal tendría una baja precisión y se decide usarla como prueba de cordura para comparar los demás modelos entrenados.
- Para cada método se evaluan 2 configuraciones de complejidad diferentes. EL obtetivo de ello era medir tanto las métricas de RECM como los tiempos de entrenamiento.
- Se probaron los métodos convencionales de *Árbol de decisión* y *Bosque aleatorio*, donde como era de esperarse, el bosque muestra unas buenas métricas de evaluación con un error relativo al precio promedio moderado y cercano al 33%.
- Se implmentaron también *algoritmos de potenciación de descenso de gradiente*, los cuales mostraron unas métricas menores al 35% en su entrenamiento. El tiempo en este caso fue el protagonista al presentar valores variados según el algoritmo.
- Teniendo en cuenta que para la empresa **Rusty Bargain** es importante y crítico tanto la calidad del algoritmo, como los tiempos de entrenamiento y predicción, se concluye que el algoritmo **LightGBM presenta RECM con un error relativo al precio medio de los vehiculos menor al 30%, con tiempos de entrenamiento de 3 a 7 segundos dependiendo de la complejidad.**
- Los modelos de CatBoost y Bosque Aleatorio también se pueden considerar en la implementación de la aplicación si los tiempos de entrenamiento y predicción tienen una prioridad media baja. Sin embargo, se podría realizar un análisis de sensibilidad para encontrar los hiperparámetros óptimos en estos algoritmos.

# Lista de control

Escribe 'x' para verificar. Luego presiona Shift+Enter

- [x]  Jupyter Notebook está abierto
- [x]  El código no tiene errores- [x]  Las celdas con el código han sido colocadas en orden de ejecución- [x]  Los datos han sido descargados y preparados- [x]  Los modelos han sido entrenados
- [x]  Se realizó el análisis de velocidad y calidad de los modelos